# Module 3: Sequential Chain

Apply **Pattern 1**: break the single-agent ceiling from Module 2 by running the same decision brief through a fixed 3-stage pipeline, Researcher → Analyst → Synthesizer, where each agent has one focused job and passes its output to the next.

**You** decide the order, in code: graph edges run the stages in the same sequence every time. In Module 4 you hand that decision to an orchestrator agent instead.

**When to use this pattern:**
- Steps have a natural, fixed order
- Each stage depends on the previous one
- You want simple, predictable, debuggable flow, with no routing decisions left to the model

**Key Strands primitive:** `GraphBuilder` (`strands.multiagent`).

**Model:** this module runs on **Amazon Nova Lite**, a smaller and cheaper model than the Claude Sonnet 4.6 default used in Modules 2 and 4. A fixed graph is what makes that work: see *Why a smaller model works here* below.

**Prerequisites:** Modules 1 and 2. This module reuses Module 2's tools and the shared decision brief.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns MarketNest financial/operational data | CLV, churn rate, revenue, top-spender metrics |
| `get_market_benchmarks(industry)` | Returns e-commerce benchmarks | Avg CLV, churn, subscription rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns competitor premium tier data | Pricing, pilot approach, adoption %, CLV lift, payback period |

> **Note:** Only the **Researcher** agent has access to these tools. The Analyst and Synthesizer work with the text handed to them by the graph: they have no tool access.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# This module pins a smaller model on purpose: Amazon Nova Lite
# (amazon.nova-lite-v1:0). In a graph the edges do the routing, so each agent only does one
# narrow job, and a lighter model is enough (see the next cell).
# max_tokens is set because Strands has no output limit on record for Nova, and Bedrock's
# own default can cut a long analysis short (MaxTokensReachedException).
# To compare, swap in the default the other modules use:
#   model = BedrockModel(model_id="global.anthropic.claude-sonnet-4-6")

import os, sys, time
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "shared"))

from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data
from decision_brief import DECISION_BRIEF

model = BedrockModel(model_id="amazon.nova-lite-v1:0", max_tokens=5000)

print(f"Model: {model.config['model_id']}")
print("Tools loaded: get_company_data | get_market_benchmarks | get_competitor_data")

### Why a smaller model works here

In a graph, **your code makes the routing decisions**: the edges say who runs, in what order, and what each agent receives. Each agent is left with one narrow job: the Researcher calls three tools, the Analyst analyzes, the Synthesizer writes. A smaller, faster, cheaper model handles that well.

In Module 4 the **model makes those decisions**: an orchestrator has to plan which specialists to call, in what order, with which arguments, and when to stop. That needs a stronger model. When we ran Module 4's orchestrator on Amazon Nova 2 Lite, it called the research specialist and then stopped with `stop_reason: malformed_tool_use`, without calling finance, legal or the writer. The graph below takes those decisions away from the model, so Nova Lite only has to do one step at a time.

**Rule of thumb:** if a smaller model struggles to choose the next step, take that choice away from it. Fix the order with a graph and keep the model for the work inside each step.

---

## Part 1: Define the Prompts

Each agent gets a **narrow system prompt** focused on exactly one role. The docstring tells the model *when* to use a tool; the system prompt tells it *what job it has*.

The prompts also describe what each stage receives. A graph node gets the original task plus the output of the nodes that point to it, so the Synthesizer sees the brief and the Analyst's output, not the raw research. That is why the Analyst is asked to cite the numbers behind its verdicts.

In [ ]:
RESEARCHER_PROMPT = '''You are a market research specialist.
Given a decision brief, gather relevant company data, market benchmarks,
and competitor intelligence using your tools.
Return structured findings: data only, no recommendations.'''

ANALYST_PROMPT = '''You are a business strategy analyst.
You receive the decision brief and the Researcher's findings. Analyze each option (A, B, C).
For each option return:
- Strengths and weaknesses
- Implementation complexity: Low / Medium / High (with one-sentence justification)
- Top 2 risks with specific mitigations
- Verdict: Proceed / Proceed with caution / Do not proceed
Cite the research numbers behind each verdict: the next stage does not see the raw research.
Return structured analysis only, no executive memo yet.'''

SYNTHESIZER_PROMPT = '''You are an executive communications specialist.
You receive the decision brief and the Analyst's option analyses (which cite the research).
Write a leadership memo:

## Decision Memo: [Title]
**Recommendation**: [one sentence: which option and why]

### Options at a Glance
| | Option A | Option B | Option C |
|---|---|---|---|
| Complexity | | | |
| Risk level | | | |
| Verdict | | | |

### Top 3 Risks & Mitigations
### Success Metrics (3-5 KPIs with targets)
### Decision Required: owner · deadline · approvers needed

Under 400 words. Be direct.'''

---

## Part 2: Create the Three Agents

`callback_handler=None` makes every agent **silent** during graph execution. The graph runs all three; we display the final output after the chain completes.

In [ ]:
researcher = Agent(
    model=model,
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=RESEARCHER_PROMPT,
    callback_handler=None,
)

analyst = Agent(model=model, system_prompt=ANALYST_PROMPT, callback_handler=None)

synthesizer = Agent(model=model, system_prompt=SYNTHESIZER_PROMPT, callback_handler=None)

print("Agents created: researcher | analyst | synthesizer (all on Nova Lite)")

---

## Part 3: Build the Graph and Run the Chain

`GraphBuilder` builds a **directed graph of agents**. Nodes are agents; edges define execution order. The graph passes each node's output to the nodes that depend on it.

For a short fixed sequence, plain function calls in your application also work (the Strands docs call this a workflow). The graph gives you more for the same few lines: an execution timeout, `execution_order`, per-node results and token usage, and room to add parallel or conditional edges later (see the optional modules in `samples/extra/`).

In [ ]:
builder = GraphBuilder()
builder.add_node(researcher,  "researcher")
builder.add_node(analyst,     "analyst")
builder.add_node(synthesizer, "synthesizer")
builder.add_edge("researcher", "analyst")      # researcher → analyst
builder.add_edge("analyst",    "synthesizer")  # analyst → synthesizer
builder.set_execution_timeout(300)             # stop a runaway graph (Strands warns when no limit is set)

print("Running: Researcher → Analyst → Synthesizer")
print("─" * 60)

t0 = time.time()
graph_result = builder.build()(DECISION_BRIEF)
elapsed = time.time() - t0

print(f"Chain complete in {elapsed:.1f}s  |  status: {graph_result.status.value}")
print("─" * 60)
print(graph_result.results["synthesizer"].result)

---

## Part 4: Inspect the Pipeline

In [ ]:
for node in graph_result.execution_order:
    print(f"=== {node.node_id.upper()} (first 300 chars) ===")
    print(str(node.result)[:300], "...")
    print()

print("Execution order:", " → ".join(n.node_id for n in graph_result.execution_order))
print()
print("What each agent saw:")
print("  Researcher:  the brief + its own tool results")
print("  Analyst:     the brief + the Researcher's output")
print("  Synthesizer: the brief + the Analyst's output")

In [ ]:
# Token usage per node: each agent has its own context, so you can see what each stage cost.
print(f"{'Stage':<14} {'Input':>8} {'Output':>8} {'Total':>8}")
print("-" * 42)
for node in graph_result.execution_order:
    u = node.result.accumulated_usage
    print(f"{node.node_id:<14} {u['inputTokens']:>8,} {u['outputTokens']:>8,} {u['totalTokens']:>8,}")
u = graph_result.accumulated_usage
print("-" * 42)
print(f"{'graph total':<14} {u['inputTokens']:>8,} {u['outputTokens']:>8,} {u['totalTokens']:>8,}")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| Sequential Chain | `GraphBuilder` defines the graph; edges fix the execution order |
| `add_node` / `add_edge` | Declarative graph definition, no manual Python orchestration |
| Output propagation | Each node gets the original task plus the output of the nodes before it |
| Focused context | Each agent sees only its inputs; per-node token usage shows what each stage cost |
| Fixed order | Your edges pick the order, not the model; Module 4 hands that choice to an orchestrator |
| Smaller model | With the routing in the graph, Nova Lite runs every stage; Module 4's orchestrator needs a stronger model |

**Strands primitive:** `GraphBuilder` (Graph)

---

## What's Next

In **Module 4: Agent-as-Tool**, an orchestrator agent decides which specialist to call and in what order, instead of graph edges. That module goes back to Claude Sonnet 4.6: planning and routing need a stronger model.

Optional: in [Parallel Fork-Join](../extra/04-parallel-fork-join/) the analysts run at the same time instead of in sequence, with the same graph primitive and parallel edges.

---

## Want a real multi-turn conversation?

```bash
cd samples/03-sequential-chain
pip install -r requirements.txt
python chat.py
```